# Mapping shorelines from satellite imagery
### A hands-on workshop notebook

**What you will do (about 50 min):**

| Step | What | Time |
|---|---|---|
| 1 | *(Demo)* Download imagery from Google Earth Engine | 
| 2 | Load the images | 
| 3 | Look at one image and compute a water index (NDWI) | 
| 4 | Extract a shoreline from one image and adjust the threshold | 
| 5 | Add a transect and measure shoreline position |
| 6 | Automatically Repeat for every image to build a time series | 
| 7 | Analyse the time series: noise, filtering, seasonality, trend | 
|  | new beach | 
| 8 | Correct for tides | 
| 9 | noise, variability, uncertainty, and other considerations | 

**How to use it:** run each cell in order (Shift+Enter). Cells marked **Your turn** are where you change something and see what happens. Cells marked *helper code* just define functions, so run them but you don't need to read them.


## 0. Setup

In [ ]:
import os

import re, glob, os, warnings
from pathlib import Path
import numpy as np
import pandas as pd

import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, signal
from scipy.ndimage import gaussian_filter
from skimage import measure, filters
import rasterio
from rasterio.warp import reproject, calculate_default_transform, Resampling

import ipywidgets

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print("Ready.")

## Settings
Everything you might need to change is in this one cell.

In [ ]:

SITE = "ewa"

# ---------- REAL DATA (only used when MODE = "folder") ----------
DATA_DIR    = f"sat_images/{SITE}"   # folder containing the .tif files (searched recursively)
FILE_FILTER = "S2" #""                    # optional: only use files whose name contains this text, e.g. "S2"


# Band positions inside each GeoTIFF (1 = first band). CHECK THESE with the
# band-inspection cell in Step 2: they depend on how the images were exported.
BANDS = dict(red=1, green=2, blue=3, nir=4, swir1=5, swir2=6)
AUTO_DETECT_BANDS = True      # if band names like 'B3' / 'B8' are stored in the file, use them instead

# How to read the date from the file name (first group = year, month, day)
DATE_REGEX = r"(\d{4})[-_]?(\d{2})[-_]?(\d{2})"

# ---------- QUALITY CONTROL ----------
MIN_VALID_FRAC = 0.50         # skip images with more than 50% missing/no-data pixels
MIN_CONTOUR_LEN_M = 100       # ignore tiny contour fragments shorter than this (metres)



## Step 1 (demo): Download imagery from Google Earth Engine
*Presenter only.* Everyone needing an Earth Engine account set up beforehand is a big time sink, so run this once yourself (before the workshop or live on screen) and share the images for the hands-on part.

This uses the [`geedownload`](https://github.com/Coastal-Research-Collaborative/geedownload) package, which downloads Landsat and Sentinel-2 imagery from Earth Engine for a polygon and date range.

**Before you run it you need:** a Google Earth Engine account and a Google Cloud project registered for Earth Engine.

> The coordinates below are the example from the package's README (Jekyll Island, Georgia, USA). Replace them with your own site. The polygon is a closed ring of `[longitude, latitude]` points.

## Step 2: Load the images
Two things happen here: each image is loaded and put on a common grid, and we compute **NDWI** (Normalised Difference Water Index) for each one:

$$\text{NDWI} = \frac{\text{Green} - \text{NIR}}{\text{Green} + \text{NIR}}$$

Water reflects green light but absorbs near-infrared (NIR), so water has **high** NDWI and land has **low** NDWI. That contrast is what lets us find the shoreline.

In [ ]:
# ======================= HELPER CODE =======================

def ndwi_from_bands(green, nir):
    s = green + nir
    with np.errstate(divide="ignore", invalid="ignore"):
        out = (green - nir) / s
    out[~np.isfinite(out) | (s <= 0)] = np.nan
    return out.astype("float32")

def mndwi(green, swir):
    s = green + swir
    with np.errstate(divide="ignore", invalid="ignore"):
        out = (green - swir) / s
    out[~np.isfinite(out) | (s <= 0)] = np.nan
    return out.astype("float32")

# ---------- Real GeoTIFFs ----------
def utm_epsg(lon, lat):
    zone = int((lon + 180)//6) + 1
    return (32600 if lat >= 0 else 32700) + zone

def make_reference_grid(path):
    # the first image defines the common grid (metres); degrees are converted to UTM
    with rasterio.open(path) as src:
        if src.crs is not None and src.crs.is_geographic:
            b = src.bounds
            dst_crs = rasterio.crs.CRS.from_epsg(utm_epsg((b.left+b.right)/2, (b.top+b.bottom)/2))
            tr, w, h = calculate_default_transform(src.crs, dst_crs, src.width, src.height, *src.bounds, resolution=10)
            return dict(crs=dst_crs, transform=tr, width=w, height=h)
        return dict(crs=src.crs, transform=src.transform, width=src.width, height=src.height)

def resolve_bands(src):
    idx = dict(BANDS)
    if AUTO_DETECT_BANDS and src.descriptions and any(src.descriptions):
        desc = [(d or "").lower() for d in src.descriptions]
        wanted = dict(blue=("blue", "b2", "sr_b2"), green=("green", "b3", "sr_b3"),
                      red=("red", "b4", "sr_b4"), nir=("nir", "b8", "sr_b5", "b8a"))
        for key, names in wanted.items():
            for i, dsc in enumerate(desc):
                if dsc in names:
                    idx[key] = i + 1
                    break
    return idx

def read_band(src, band, grid):
    arr = src.read(band).astype("float32")
    if src.nodata is not None:
        arr[arr == src.nodata] = np.nan
    arr[arr == 0] = np.nan
    dst = np.full((grid["height"], grid["width"]), np.nan, dtype="float32")
    reproject(source=arr, destination=dst, src_transform=src.transform, src_crs=src.crs,
              dst_transform=grid["transform"], dst_crs=grid["crs"],
              resampling=Resampling.bilinear, src_nodata=np.nan, dst_nodata=np.nan)
    return dst

def parse_date(name):
    for m in re.finditer(DATE_REGEX, name):
        try:
            return pd.Timestamp(int(m.group(1)), int(m.group(2)), int(m.group(3)))
        except Exception:
            continue
    return None

import re
def parse_time_utc(name):
    """Full acquisition timestamp from a name like S2_20160414_211919.tif (HHMMSS). NaT if there is no time."""
    m = re.search(r"(\d{8})[_T-]?(\d{6})", name)
    return pd.to_datetime(m.group(1) + m.group(2), format="%Y%m%d%H%M%S") if m else pd.NaT


def load_scenes_from_folder(data_dir, with_rgb_first=True):
    files = sorted(glob.glob(os.path.join(data_dir, "**", "*.tif"), recursive=True) +
                   glob.glob(os.path.join(data_dir, "**", "*.tiff"), recursive=True))
    if FILE_FILTER:
        files = [f for f in files if FILE_FILTER in os.path.basename(f)]
    if not files:
        raise FileNotFoundError(f"No .tif files found in '{data_dir}'. Check DATA_DIR / FILE_FILTER.")
    grid = make_reference_grid(files[0])
    res = abs(grid["transform"].a)
    scenes, skipped = [], []
    for f in files:
        date = parse_date(os.path.basename(f))
        if date is None:
            skipped.append((os.path.basename(f), "no date in file name")); continue
        try:
            with rasterio.open(f) as src:
                idx = resolve_bands(src)
                g = read_band(src, idx["green"], grid)
                n = read_band(src, idx["nir"], grid)
                # swir = read_band(src, idx["swir1"], grid)
                rgb = None
                if with_rgb_first:
                    rgb = np.dstack([read_band(src, idx[k], grid) for k in ("red", "green", "blue")])
        except Exception as e:
            skipped.append((os.path.basename(f), str(e)[:80])); continue
        ndwi = ndwi_from_bands(g, n)
        # mndwi = mndwi(g, swir)
        scenes.append(dict(date=date, name=os.path.basename(f), ndwi=ndwi, 
                            time_utc=parse_time_utc(os.path.basename(f)),
                           valid_frac=float(np.isfinite(ndwi).mean()), rgb=rgb, green=g, nir=n))
    best = {}
    for s in scenes:
        if s["date"] not in best or s["valid_frac"] > best[s["date"]]["valid_frac"]:
            best[s["date"]] = s
    scenes = [best[d] for d in sorted(best)]

    # NEW: drop scenes that cover too little of the site
    n_all = len(scenes)
    scenes = [s for s in scenes if s["valid_frac"] >= MIN_VALID_FRAC]
    print(f"Kept {len(scenes)} of {n_all} scenes "
          f"(dropped {n_all - len(scenes)} with < {MIN_VALID_FRAC:.0%} valid pixels)")

    if skipped:
        print(f"Skipped {len(skipped)} file(s):", *skipped[:5], sep="\n  ")
    return scenes, dict(res=res, ny=grid["height"], nx=grid["width"])


print("Helper functions defined.")

In [ ]:
############ LOAD DATA #################
scenes, GRID, TRUTH = *load_scenes_from_folder(DATA_DIR), None

RES = GRID["res"]
print(f"{len(scenes)} images loaded | pixel size {RES:g} m | grid {GRID['ny']} x {GRID['nx']} pixels")
print(f"first: {scenes[0]['date']:%Y-%m-%d}   last: {scenes[-1]['date']:%Y-%m-%d}")

# save loaded scene names
names = [scenes[i]['name'] for i in range(len(scenes))]

In [ ]:
# are 'green' and 'NIR' really what we think they are?
# Water should look DARK in NIR and BRIGHTER in green. If it doesn't, fix BANDS in Settings.

IMG = 23
f0 = sorted(glob.glob(os.path.join(DATA_DIR, "**", f"{names[IMG]}"), recursive=True))[0]

with rasterio.open(f0) as src:
    print("File:", os.path.basename(f0), "| bands:", src.count, "| CRS:", src.crs)
    print("Band descriptions:", src.descriptions)
    print("Bands being used:", resolve_bands(src))
    fig, axs = plt.subplots(1, src.count, figsize=(3*src.count, 3))
    for i, ax in enumerate(np.atleast_1d(axs)):
        a = src.read(i + 1).astype(float); a[a == 0] = np.nan
        ax.imshow(a, cmap="gray", vmin=np.nanpercentile(a, 2), vmax=np.nanpercentile(a, 98))
        ax.set_title(f"band {i+1}"); ax.axis("off")
    plt.show()


## Step 3: Look at one image
Pick an image and look at it three ways: true colour, the green and NIR bands, and NDWI.

In [ ]:
IMG = 3        # <- Your turn: try different images (0 to len(scenes)-1). Can you find a cloudy one?

sc = scenes[IMG]
ext = (0, GRID["nx"]*RES, GRID["ny"]*RES, 0)     # axes in metres, y pointing down
fig, axs = plt.subplots(1, 3, figsize=(15, 4))

if sc["rgb"] is not None:
    rgb = sc["rgb"].copy()
    lo, hi = np.nanpercentile(rgb, 2), np.nanpercentile(rgb, 98)
    axs[0].imshow(np.clip(np.nan_to_num((rgb - lo)/(hi - lo)), 0, 1), extent=ext)
    axs[0].set_title("True colour")
else:
    axs[0].imshow(sc["nir"], cmap="gray", extent=ext); axs[0].set_title("NIR band")
    
axs[1].imshow(sc["green"], cmap="gray", extent=ext); axs[1].set_title("Green band")
im = axs[2].imshow(sc["ndwi"], cmap="RdYlBu", vmin=-1, vmax=1, extent=ext)
axs[2].set_title("NDWI  (blue = water, red = land)")
plt.colorbar(im, ax=axs[2], fraction=0.046)
for ax in axs: ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)"); ax.grid(False)
fig.suptitle(f"{sc['name']}   |   {sc['date']:%d %b %Y}   |   {sc['valid_frac']*100:.0f}% valid pixels")
plt.tight_layout(); plt.show()

**Questions to discuss**
- Where is the shoreline in the NDWI image? What value of NDWI would you pick as the boundary between land and water?
- Does the NDWI image look the same for every date? Why might it not?

## Step 4: Extract a shoreline from one image
**Idea:** a shoreline is just the **contour line** in the NDWI image where values cross a **threshold**. 

Which threshold? We can let **Otsu's method** pick one automatically: it looks at the histogram of NDWI values and finds the split that best separates two groups (water and not-water).

In [ ]:
# ======================= HELPER CODE =======================
def nan_gaussian(a, sigma):
    # Gaussian blur that ignores missing pixels
    m = np.isfinite(a).astype(float)
    num = gaussian_filter(np.where(m > 0, a, 0.0), sigma)
    den = gaussian_filter(m, sigma)
    out = np.where(den > 1e-3, num/np.maximum(den, 1e-3), np.nan)
    out[m == 0] = np.nan
    return out

def contour_length(c):
    return float(np.hypot(*np.diff(c, axis=0).T).sum())

def extract_shoreline(ndwi, res, threshold=None, smooth_sigma=0.0, min_len_m=None, min_valid=None):
    # returns dict(threshold, contours) or None if the image fails quality control
    min_len_m = MIN_CONTOUR_LEN_M if min_len_m is None else min_len_m
    min_valid = MIN_VALID_FRAC if min_valid is None else min_valid
    valid = np.isfinite(ndwi)
    if valid.mean() < min_valid:
        return None
    img = nan_gaussian(ndwi, smooth_sigma) if smooth_sigma else ndwi
    thr = filters.threshold_otsu(ndwi[valid]) if threshold is None else float(threshold)
    raw = measure.find_contours(np.where(valid, img, np.nanmedian(img)), thr, mask=valid)
    contours = []
    for c in raw:
        xy = np.column_stack([(c[:, 1] + 0.5)*res, (c[:, 0] + 0.5)*res])   # (col,row) -> metres
        if contour_length(xy) >= min_len_m:
            contours.append(xy)
    return dict(threshold=thr, contours=contours)

print("extract_shoreline() defined.")

In [ ]:
# IMG = 40
sc = scenes[IMG]

result = extract_shoreline(sc["ndwi"], RES)           # threshold=None  ->  Otsu

fig, axs = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw=dict(width_ratios=[1.3, 1]))
axs[0].imshow(sc["ndwi"], cmap="RdYlBu", vmin=-1, vmax=1, extent=ext)
for c in result["contours"]:
    axs[0].plot(c[:, 0], c[:, 1], "k-", lw=2)
axs[0].set_title(f"Shoreline at NDWI = {result['threshold']:.2f}"); axs[0].grid(False)
axs[0].set_xlabel("x (m)"); axs[0].set_ylabel("y (m)")

vals = sc["ndwi"][np.isfinite(sc["ndwi"])]
axs[1].hist(vals, bins=80, color="steelblue")
axs[1].axvline(result["threshold"], color="k", lw=2, label=f"Otsu threshold = {result['threshold']:.2f}")
axs[1].set_xlabel("NDWI"); axs[1].set_ylabel("pixels"); axs[1].legend(); axs[1].set_title("Histogram of NDWI")
plt.tight_layout(); plt.show()

**Your turn: move the threshold.** Drag the slider and watch the line move. Then try the same on a *cloudy* image (change `IMG` in Step 3 or the image slider below).

- How far does the line move (in metres) between NDWI = 0.0 and 0.4?
- Is there a "right" answer? What does this tell you about how precise a satellite shoreline can be?

In [ ]:
import ipywidgets as widgets
from IPython.display import display

def show_threshold(img_index=IMG, threshold=0.0, use_otsu=False):
    s = scenes[img_index]
    r = extract_shoreline(s["ndwi"], RES, threshold=None if use_otsu else threshold)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.imshow(s["ndwi"], cmap="RdYlBu", vmin=-1, vmax=1, extent=ext)
    if r is None:
        ax.set_title("Image failed quality control (too many missing pixels)")
    else:
        for c in r["contours"]:
            ax.plot(c[:, 0], c[:, 1], "k-", lw=2)
        ax.set_title(f"{s['date']:%d %b %Y}  |  threshold = {r['threshold']:.2f}"
                     f"{'  (Otsu)' if use_otsu else ''}")
    ax.grid(False); ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")
    plt.show()

widgets.interact(show_threshold,
                 img_index=widgets.IntSlider(value=IMG, min=0, max=len(scenes)-1, description="image"),
                 threshold=widgets.FloatSlider(value=0.0, min=-0.6, max=0.8, step=0.02,
                                               description="threshold", continuous_update=False),
                 use_otsu=widgets.Checkbox(value=False, description="use Otsu"));

In [ ]:
plt.close('all')

## Step 5: Add a transect and measure shoreline position
A **transect** is a line drawn across the beach, perpendicular to the shore. Where the shoreline crosses the transect gives a single number: the **cross-shore distance** from the transect's landward start point. If that distance increases the beach is **growing** (accretion); if it decreases the shoreline is **retreating** (erosion).

We define the transect in metres on the image's grid (origin at the **top-left corner**, `x` to the right, `y` downward), going **from land to sea**: `((x_start, y_start), (x_end, y_end))`.

In [ ]:
# ---- Your turn: define the transect ----------------------------------------------------
# Land-to-sea line in metres. Use the gridlines in the plot below to pick coordinates.
TRANSECT = ((300, 500), (500, 700))       # default suits the synthetic coastline
TRANSECT = ((300, 500), (600, 750))       # default suits the synthetic coastline
# TRANSECT = ((1500, 2000), (1000, 1500))       # default suits the synthetic coastline
# --------------------------------------------------------------------------------------

sc = scenes[IMG]
fig, ax = plt.subplots(figsize=(9, 6))
ax.imshow(sc["ndwi"], cmap="RdYlBu", vmin=-1, vmax=1, extent=ext)
ax.annotate("", xy=TRANSECT[1], xytext=TRANSECT[0],
            arrowprops=dict(arrowstyle="-|>", color="k", lw=2.5))
ax.plot(*TRANSECT[0], "ko"); ax.text(TRANSECT[0][0], TRANSECT[0][1]-25, "start (land)", ha="center")
ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)"); ax.set_title("Transect (land to sea)")
ax.grid(True, color="k", alpha=0.25)
plt.show()

In [ ]:
# ======================= HELPER CODE =======================
def transect_intersections(contours, p0, p1):
    # distances (m) from p0 along the transect where any contour crosses it
    p0 = np.asarray(p0, float); d = np.asarray(p1, float) - p0; L = float(np.hypot(*d))
    out = []
    for c in contours:
        a, e = c[:-1], c[1:] - c[:-1]
        w = a - p0
        denom = d[0]*e[:, 1] - d[1]*e[:, 0]
        with np.errstate(divide="ignore", invalid="ignore"):
            t = (w[:, 0]*e[:, 1] - w[:, 1]*e[:, 0]) / denom
            s = (w[:, 0]*d[1] - w[:, 1]*d[0]) / denom
        ok = (np.abs(denom) > 1e-12) & (t >= 0) & (t <= 1) & (s >= 0) & (s <= 1)
        out.extend((t[ok]*L).tolist())
    return np.array(sorted(out))
print("transect_intersections() defined.")

In [ ]:
sc = scenes[IMG]
r = extract_shoreline(sc["ndwi"], RES)
cross = transect_intersections(r["contours"], *TRANSECT)
print("Shoreline crosses the transect at distance(s) (m):", np.round(cross, 1))

## dummy check in case theres no intersects
if len(cross) == 0:
    raise SystemExit("No crossing on this image. Try another IMG.")

d = cross[0]
p0 = np.array(TRANSECT[0], float); u = np.array(TRANSECT[1], float) - p0; u /= np.hypot(*u)
pt = p0 + u*d

fig, ax = plt.subplots(figsize=(9, 6))
ax.imshow(sc["ndwi"], cmap="RdYlBu", vmin=-1, vmax=1, extent=ext)
for c in r["contours"]: ax.plot(c[:, 0], c[:, 1], "k-", lw=2)
ax.plot(*zip(*TRANSECT), "w--", lw=2)
ax.plot(*pt, "o", ms=12, mfc="yellow", mec="k")
ax.set_title(f"Shoreline position on this date = {d:.0f} m from the transect start")
ax.grid(False); ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")
plt.show()

**That one number is one point on a time series.** Now we repeat it for every image.

## Step 6: Loop over all images, build a time series
Same code, wrapped in a loop. Two safeguards:
1. Images that fail quality control (too many missing pixels) are skipped.
2. If the shoreline crosses the transect more than once (e.g. around a cloud), we keep the crossing **closest to the typical position** (median of positions obtaiend so far).

In [ ]:
def build_timeseries(scenes, transect, threshold=None, smooth_sigma=0.0):
    rows = []
    for s in scenes:
        r = extract_shoreline(s["ndwi"], RES, threshold=threshold, smooth_sigma=smooth_sigma)
        cands = np.array([]) if r is None else transect_intersections(r["contours"], *transect)
        rows.append(dict(date=s["date"], time_utc=s["time_utc"], threshold=np.nan if r is None else r["threshold"],
                         n_crossings=len(cands), cands=cands, valid_frac=s["valid_frac"]))
    singles = [r["cands"][0] for r in rows if r["n_crossings"] == 1]
    ref = np.median(singles) if singles else np.nan          # "typical" shoreline position
    for r in rows:
        r["dist_m"] = r["cands"][np.argmin(np.abs(r["cands"] - ref))] if r["n_crossings"] else np.nan
    df = pd.DataFrame(rows).drop(columns="cands")
    return df

ts_raw = build_timeseries(scenes, TRANSECT)
print(f"{ts_raw['dist_m'].notna().sum()} of {len(ts_raw)} images gave a shoreline position")
print(f"{(ts_raw['n_crossings'] > 1).sum()} images had more than one crossing, "
      f"{(ts_raw['n_crossings'] == 0).sum()} had none")
ts_raw.head()

In [ ]:
# All the extracted shorelines, coloured by date (zoomed in around the transect)
fig, ax = plt.subplots(figsize=(9, 6))
cmap = plt.cm.viridis
t0, t1 = scenes[0]["date"], scenes[-1]["date"]
for s in scenes:
    r = extract_shoreline(s["ndwi"], RES)
    if r is None: continue
    col = cmap((s["date"] - t0)/(t1 - t0))
    for c in r["contours"]: ax.plot(c[:, 0], c[:, 1], color=col, lw=1, alpha=0.8)
ax.plot(*zip(*TRANSECT), "k--", lw=2, label="transect")
ax.plot(*TRANSECT[0], "ko"); ax.text(TRANSECT[0][0], TRANSECT[0][1], "  start (land)", va="bottom")
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 1))
cb = plt.colorbar(sm, ax=ax, fraction=0.04); cb.set_ticks([0, 1]); cb.set_ticklabels([f"{t0:%b %Y}", f"{t1:%b %Y}"])

# Zoom on the transect, whichever way it points. (Using start -> end directly as the axis limits
# flips the plot whenever the transect runs towards smaller x, so use min/max instead.)
(xa, ya), (xb, yb) = TRANSECT
pad = 0.5*max(abs(xb - xa), abs(yb - ya), 100)
ax.set_xlim(min(xa, xb) - pad, max(xa, xb) + pad)
ax.set_ylim(max(ya, yb) + pad, min(ya, yb) - pad)       # y points down, like the image
ax.set_aspect("equal"); ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)"); ax.legend()
ax.set_title("Every shoreline, coloured by date")
plt.show()

In [ ]:
# Save it: this is also the 'checkpoint' file you can share so nobody gets stuck
out = ts_raw[["date", "time_utc", "dist_m"]].copy()
out.to_csv(f"{SITE}_shoreline_timeseries.csv", index=False)
print(f"Saved {SITE}_shoreline_timeseries.csv")

## Step 7: Analyse the time series
Now we have one number per image. Real shoreline time series are **noisy**, and the analysis is about separating signal from noise. Everything below uses the table `ts` (date, shoreline distance in metres).

> **Shortcut:** if your loop in Step 6 didn't work, set `USE_CHECKPOINT = True` in Settings and point `CHECKPOINT_CSV` at the shared file.

In [ ]:
USE_CHECKPOINT = True
####### If something is broken, use USE_CHECKPOINT = True above to load a pre-saved csv
if USE_CHECKPOINT == True:
    ts_raw = pd.read_csv(rf"{SITE}_shoreline_timeseries.csv", parse_dates=["date", "time_utc"])
    print("Loaded checkpoint file.")
#############

ts = ts_raw[["date", "time_utc", "dist_m"]].copy()

ts = ts.dropna().sort_values("date").reset_index(drop=True)
ts["t_yr"] = (ts["date"] - ts["date"].min()).dt.days / 365.25      # years since first image
ts["yfrac"] = (ts["date"].dt.dayofyear - 1) / 365.25      # time of year: 0 = 1 Jan
print(f"{len(ts)} observations spanning {ts['t_yr'].max():.1f} years")

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(ts["date"], ts["dist_m"], "o-", ms=4, lw=0.8, color="tab:blue")
ax.set_ylabel("Shoreline position (m from transect start)"); ax.set_title("Raw satellite shoreline time series")
plt.show()

### 7a. Noise and outliers
Some points are clearly wrong (clouds, bad thresholds), and some are right but look wrong (storms). We flag points that sit far from the **running median** of their neighbours, using a robust noise estimate (MAD - median absolute deviation).

In [ ]:
OUTLIER_K = 3.0        # <- Your turn: flag points more than K "robust standard deviations" from the running median

s = ts.set_index("date")["dist_m"]
run_med = s.rolling(7, center=True, min_periods=3).median()
resid = (s - run_med).values
noise_sigma = 1.4826*np.median(np.abs(resid - np.median(resid)))      # robust std of the residuals
ts["outlier"] = np.abs(resid) > OUTLIER_K*noise_sigma
clean = ts[~ts["outlier"]].copy()

gaps = ts["date"].diff().dt.days.dropna()
print(f"Typical noise level (robust std): {noise_sigma:.1f} m   (pixel size is {RES:g} m)")
print(f"Flagged {ts['outlier'].sum()} of {len(ts)} points as outliers")
print(f"Sampling: median gap {gaps.median():.0f} days, longest gap {gaps.max():.0f} days")

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(ts["date"], run_med.values, "-", color="gray", lw=1.5, label="running median (7 points)")
ax.plot(clean["date"], clean["dist_m"], "o", ms=5, color="tab:blue", label="kept")
ax.plot(ts.loc[ts["outlier"], "date"], ts.loc[ts["outlier"], "dist_m"], "rx", ms=9, mew=2, label="flagged")
ax.set_ylabel("Shoreline position (m)"); ax.legend(); ax.set_title("Outlier flagging")
plt.show()

### closer look at cleaned up timeseies 

In [ ]:
# calculate new running med
s = clean.set_index("date")["dist_m"]
run_med = s.rolling(7, center=True, min_periods=3).median()

fig, ax = plt.subplots(figsize=(11, 4))
# ax.plot(clean["date"], run_med.values, "-", color="gray", lw=1.5, label="running median (7 points)")
ax.plot(clean["date"], clean["dist_m"], "o", ms=5, color="tab:blue", label="kept")
ax.set_ylabel("Shoreline position (m)"); ax.legend(); ax.set_title("clean data")
plt.show()

**Discuss:** are all the flagged points really errors? Does the clean time-series look clean? (hint hint.. most likely not at all...)

### 7b. Filtering: smoothing out the noise
Three common ways to look at a noisy timeseries

In [ ]:
WINDOW_DAYS = 90     # <- try 30, 90, 180. What happens to the signal?

s_clean = clean.set_index("date")["dist_m"]

# simple monthtly median
monthly = s_clean.resample("MS").median()

# rolling 3 month window
roll_med = s_clean.rolling(f"{WINDOW_DAYS}D", center=True, min_periods=3).median()

# savgol_filter 
GRID_DAYS = 15      # Savitzky-Golay needs evenly spaced data: average the images onto a grid this many days apart
SG_ORDER = 2       # polynomial fitted inside each window (2 = a parabola)
reg = s_clean.resample("15D").mean().interpolate(limit_area="inside")          # regular 15-day grid
win = max(5, (int(WINDOW_DAYS/GRID_DAYS) // 2)*2 + 1)                                   # odd window length
win = min(win, len(reg) - (1 - len(reg) % 2))
sg = pd.Series(signal.savgol_filter(reg.bfill().ffill().values, win, SG_ORDER), index=reg.index)


fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(s_clean.index, s_clean.values, ".", color="lightgray", ms=8, label="cleaned data")
ax.plot(roll_med.index, roll_med.values, lw=2, label=f"rolling median ({WINDOW_DAYS} d)")
ax.plot(sg.index, sg.values, lw=2, label=f"Savitzky-Golay (window {win} x 15 d)")
ax.plot(monthly.index, monthly.values, "s-", ms=4, lw=1, label="monthly median")
ax.set_ylabel("Shoreline position (m)"); ax.legend(ncol=2); ax.set_title("Three ways to smooth")
plt.show()

**plain moving average vs Savitzky-Golay ?** A moving average is the same idea with a *flat line* (order 0) in each window, so it clips the top of every peak and the bottom of every dip. A parabola can bend with the data, so Savitzky-Golay keeps peak heights much better (a storm cut, a seasonal maximum), at the price of letting a bit more noise through. A wider window means smoother; a higher order means more flexible.

### 7c. Seasonality and trend
Beaches usually have a **seasonal cycle** (e.g. erode in the stormy season, rebuild in calm months).
We can set up a simple sinusoidal trend to fit the seasonal variability 

Altough beaches typically erode fast and rebuild slower (see https://doi.org/10.1029/2024JF007836 for more info), this is a great first go at seasonality


### harmonic fit of timeseries 101 

$$y(t) = a + b\,t + c\sin(2\pi t) + d\cos(2\pi t)$$

- $y$ is the shoreline position, $t$ is the **years since the first image** (this is what the trend uses), and $f$ is **the time of year** (0 = 1 January, 0.5 = early July, 1 = 31 December).
- $a$ is the average position, $b$ is the long-term **trend** (m/yr).
- $c$ and $d$ together describe the **seasonal wave**. Its height (amplitude) is $A=\sqrt{c^2+d^2}$.

**Why both a sine *and* a cosine?** We want a wave that repeats once a year, $A\sin(2\pi f+\varphi)$, with an unknown **height** $A$ and an unknown **timing** $\varphi$ (the "phase": when in the year the beach is furthest out). The trouble is that $\varphi$ sits *inside* the sine, so the model is not a straight-line fit in $\varphi$ and least squares cannot solve for it directly. The angle-addition identity gets around this:

$$A\sin(2\pi f+\varphi) \;=\; \underbrace{A\cos\varphi}_{c}\,\sin(2\pi f) \;+\; \underbrace{A\sin\varphi}_{d}\,\cos(2\pi f)$$

Now $c$ and $d$ are ordinary multipliers, so one least-squares fit finds both.

In [ ]:
def plot_harmonic(c=2.0, d=2.0):
    f = np.linspace(0, 1, 400)
    s_part, c_part = c*np.sin(2*np.pi*f), d*np.cos(2*np.pi*f)
    A = np.hypot(c, d); phi = np.arctan2(d, c)
    f_peak = ((np.pi/2 - phi)/(2*np.pi)) % 1
    peak_day = pd.Timestamp("2021-01-01") + pd.Timedelta(days=f_peak*365.25)
    fig, ax = plt.subplots(figsize=(9, 3.6))
    ax.plot(f*12, s_part, "--", color="tab:orange", label=f"c sin(2$\\pi$f)   c = {c:.1f}")
    ax.plot(f*12, c_part, "--", color="tab:green", label=f"d cos(2$\\pi$f)   d = {d:.1f}")
    ax.plot(f*12, s_part + c_part, "k-", lw=2.5, label="sum = the seasonal cycle")
    if A > 0: ax.axvline(f_peak*12, color="k", ls=":", lw=1)
    ax.set_xticks(np.arange(12) + 0.5); ax.set_xticklabels(list("JFMAMJJASOND")); ax.set_xlim(0, 12); ax.set_ylim(-8, 8)
    ax.set_ylabel("Shoreline position (m)"); ax.legend(loc="lower left", fontsize=8, ncol=3)
    ax.set_title(f"amplitude $\\sqrt{{c^2+d^2}}$ = {A:.1f} m;  highest around {peak_day:%d %b}" if A > 0 else "no seasonal cycle")
    plt.show()

widgets.interact(plot_harmonic,
                 c=widgets.FloatSlider(value=2.0, min=-5, max=5, step=0.5, description="c (sine)", continuous_update=False),
                 d=widgets.FloatSlider(value=2.0, min=-5, max=5, step=0.5, description="d (cosine)", continuous_update=False));

### Seasonality and trend of our measured beach

In [ ]:
def fit_harmonic(t, yf, y):
    """Fit y = a + b*t + c*sin(2*pi*yf) + d*cos(2*pi*yf).
    t  = years since the first image (for the trend)
    yf = time of year, 0 = 1 Jan (for the seasonal wave)"""
    X = np.column_stack([np.ones_like(t), t, np.sin(2*np.pi*yf), np.cos(2*np.pi*yf)])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    res = y - X @ beta
    dof = len(y) - X.shape[1]
    cov = (res @ res / dof) * np.linalg.inv(X.T @ X)
    se = np.sqrt(np.diag(cov))
    return beta, se, dof, X @ beta, res

def linear_trend(t, y):
    lr = stats.linregress(t, y)
    return lr.slope, stats.t.ppf(0.975, len(t) - 2)*lr.stderr

t_all, y_all = ts["t_yr"].values, ts["dist_m"].values
t_c, yf_c, y_c = clean["t_yr"].values, clean["yfrac"].values, clean["dist_m"].values

beta, se, dof, fit, res_h = fit_harmonic(t_c, yf_c, y_c)
amp = np.hypot(beta[2], beta[3])
rows = [("Linear, all points (incl. outliers)", *linear_trend(t_all, y_all)),
        ("Linear, outliers removed",            *linear_trend(t_c, y_c)),
        ("Trend + season, outliers removed",    beta[1], stats.t.ppf(0.975, dof)*se[1])]
table = pd.DataFrame(rows, columns=["method", "trend (m/yr)", "+/- 95% CI"]).round(2)
print(table.to_string(index=False))
print(f"Fitted seasonal amplitude: {amp:.1f} m   |   residual scatter (std): {res_h.std():.1f} m")

# when in the year is the beach furthest out?
phi = np.arctan2(beta[3], beta[2])
f_peak = ((np.pi/2 - phi)/(2*np.pi)) % 1
peak = pd.Timestamp("2021-01-01") + pd.Timedelta(days=f_peak*365.25)
print(f"Seasonal peak (beach furthest out): around {peak:%d %b}")

# smooth curve for plotting: needs both the years-since-start (trend) and the time of year (season)
dates_tt = pd.date_range(ts["date"].min(), ts["date"].max(), periods=600)
tt  = ((dates_tt - ts["date"].min()).days.values) / 365.25
yft = (dates_tt.dayofyear.values - 1) / 365.25
Xs = np.column_stack([np.ones_like(tt), tt, np.sin(2*np.pi*yft), np.cos(2*np.pi*yft)])



fig, axs = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axs[0].plot(clean["date"], y_c, "o", ms=4, color="tab:blue", label="data")
axs[0].plot(dates_tt, Xs @ beta, "k-", lw=2, label="trend + season")
axs[0].plot(dates_tt, beta[0] + beta[1]*tt, "r--", lw=2, label=f"trend only ({beta[1]:.1f} m/yr)")
axs[0].set_ylabel("Position (m)"); axs[0].legend(); axs[0].set_title("Fit")
deseason = y_c - (beta[2]*np.sin(2*np.pi*yf_c) + beta[3]*np.cos(2*np.pi*yf_c))
axs[1].plot(clean["date"], deseason, "o-", ms=4, lw=0.7, color="tab:green")
axs[1].plot(dates_tt, beta[0] + beta[1]*tt, "r--", lw=2)
axs[1].set_ylabel("Position, season removed (m)"); axs[1].set_title("Seasonal cycle removed")
plt.tight_layout(); plt.show()

In [ ]:
# The seasonal cycle by month of the year (after removing the long-term trend)
detr = clean["dist_m"] - (beta[0] + beta[1]*clean["t_yr"])
by_month = [detr[clean["date"].dt.month == m].values for m in range(1, 13)]
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.boxplot(by_month, positions=range(1, 13), widths=0.6)
ax.axhline(0, color="gray", lw=1)
ax.set_xticks(range(1, 13)); ax.set_xticklabels(list("JFMAMJJASOND"))
ax.set_ylabel("Detrended position (m)"); ax.set_title("Seasonal cycle: which months are the beach furthest in / out?")
plt.show()

**Discuss:** the confidence interval above assumes each point is independent. Shoreline points close in time are *not* independent (a beach that is eroded today is likely eroded next week). Does that make the real uncertainty bigger or smaller than the table says?

A useful rule of thumb: a trend is only meaningful if the **total change over the record** (trend x record length) is clearly larger than the **noise level** from 7a. Compare the numbers printed below.



## MAP A NEW BEACH! 



### Wait - There's MORE! 

## Step 8: Correct for the tide
A satellite sees the waterline **at the instant it passes over**. As the tide rises and falls, the waterline slides up and down the beach face. If the beach has slope $\tan\beta$ (rise over run, e.g. 0.05 = 1:20), a tide height $z$ above the reference level moves the waterline **landward** by $z/\tan\beta$. So to get the shoreline we would have seen at the reference level (here mean sea level) we move each measured position **seaward** by that amount:

$$x_\text{corrected} = x_\text{measured} + \frac{z_\text{tide} - z_\text{ref}}{\tan\beta}$$

On a 1:20 beach, a tide only 0.5 m above the reference moves the waterline by **10 m, a whole Sentinel-2 pixel**. That is why tide is often the biggest noise source in a satellite shoreline record.

Three things we need: **(1)** the tide height at the moment of each image, **(2)** the beach slope, **(3)** to apply the equation.

**Where do the tide heights come from?**
- **A tide gauge** (`"noaa"`): water levels measured at a station, here Honolulu. It includes everything that moves the water (tide, storm surge, seasonal sea-level changes), but only at one place.
- **A tide model** (`"eo_tides"`): a global model that predicts the astronomical tide anywhere on Earth. [`eo-tides`](https://geoscienceaustralia.github.io/eo-tides/) runs the models for you. **EOT20** is free (CC BY 4.0, about 2 GB to download once); FES2014/FES2022 need an AVISO+ registration. Global model files are big and slow on a laptop, so `eo_tides.utils.clip_models(input_directory="tide_models/", output_directory="tide_models_clipped/", bbox=(left, bottom, right, top))` cuts them down to your region once (the docs say this can speed things up more than 10x). **Presenter:** clip once and share the small folder, rather than having everyone download 2 GB. Set `TIDE_MODEL_DIR` to that folder.
- **Your own file** (`"csv"`): columns `time_utc`, `height_m`.

In [ ]:
# ======================= HELPER CODE =======================
import json, urllib.request, urllib.parse

NOAA_API = "https://api.tidesandcurrents.noaa.gov/api/prod/datagetter"

def acquisition_times(dates):
    # time each image was taken (UTC): from the file name if it has one, else DEFAULT_UTC_TIME on that date
    by_date = {s["date"]: s.get("time_utc") for s in scenes}
    default = pd.Timedelta(f"{DEFAULT_UTC_TIME}:00")
    return pd.DatetimeIndex([by_date.get(d) if by_date.get(d) is not None else d + default for d in dates])

def fetch_noaa_tide(station, start, end, product="hourly_height", datum="MSL", cache_dir="tide_cache"):
    # Water level (m) from a NOAA CO-OPS tide station, indexed by UTC time. The API allows at most one year of hourly data per
    # request, so we ask year by year. Results are saved to disk so each station/period is only downloaded once.
    os.makedirs(cache_dir, exist_ok=True)
    cache = Path(cache_dir) / f"noaa_{station}_{product}_{datum}_{start:%Y%m%d}_{end:%Y%m%d}.csv"
    if cache.exists():
        d = pd.read_csv(cache, parse_dates=["time_utc"])
        return d.set_index("time_utc")["height_m"]
    parts, t = [], start
    while t <= end:
        t2 = min(t + pd.Timedelta(days=360), end)
        params = dict(product=product, application="coastal_workshop", begin_date=f"{t:%Y%m%d}", end_date=f"{t2:%Y%m%d}",
                      datum=datum, station=station, time_zone="gmt", units="metric", format="json")
        if product == "predictions":
            params["interval"] = "h"
        url = NOAA_API + "?" + urllib.parse.urlencode(params)
        try:
            with urllib.request.urlopen(url, timeout=60) as r:
                js = json.loads(r.read().decode())
        except Exception as e:
            raise RuntimeError(f"Could not get tide data from NOAA ({e}). Check your internet connection, or set TIDE_SOURCE = 'csv'.")
        rows = js.get("data") or js.get("predictions") or []
        if rows:
            parts.append(pd.DataFrame({"time_utc": pd.to_datetime([r["t"] for r in rows]),
                                       "height_m": pd.to_numeric([r["v"] for r in rows], errors="coerce")}))
        elif "error" in js:
            print(f"  NOAA ({t:%Y-%m-%d} to {t2:%Y-%m-%d}): {js['error'].get('message')}")
        t = t2 + pd.Timedelta(days=1)
    if not parts:
        raise RuntimeError("NOAA returned no tide data. Check TIDE_STATION, or try TIDE_PRODUCT = 'predictions'.")
    d = pd.concat(parts).dropna().drop_duplicates("time_utc").sort_values("time_utc")
    d.to_csv(cache, index=False)
    return d.set_index("time_utc")["height_m"]

def interp_tide(series, times, max_bracket_hours=2.5):
    # tide height at the image times, by linear interpolation; NaN where there is no measurement close by
    x = series.index.values.astype("datetime64[ns]").astype("int64")
    q = pd.DatetimeIndex(times).values.astype("datetime64[ns]").astype("int64")
    out = np.interp(q, x, series.values.astype(float), left=np.nan, right=np.nan)
    i = np.clip(np.searchsorted(x, q), 1, len(x) - 1)
    out[(x[i] - x[i - 1]) / 3.6e12 > max_bracket_hours] = np.nan
    return out

def image_centre_lonlat():
    # lon/lat of the middle of the first image (used to ask a tide model for the tide)
    f = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.tif*"), recursive=True))
    f = [p for p in f if FILE_FILTER in os.path.basename(p)]
    with rasterio.open(f[0]) as src:
        b = src.bounds
        lon, lat = rasterio.warp.transform(src.crs, "EPSG:4326", [(b.left + b.right)/2], [(b.top + b.bottom)/2])
    return lon[0], lat[0]

def model_tide_series(times, lonlat, model, directory):
    ensure_pkg("eo-tides", "eo_tides")
    from eo_tides.model import model_tides
    out = model_tides(x=lonlat[0], y=lonlat[1], time=pd.DatetimeIndex(times), model=model, directory=directory).reset_index()
    return pd.Series(out["tide_height"].values, index=pd.DatetimeIndex(out["time"]))


## attempt to install eo tides (it might break, if not happy tide modeling!)
def ensure_pkg(pkg, import_name=None):
    import importlib, subprocess, sys
    try:
        importlib.import_module(import_name or pkg)
    except ImportError:
        print(f"Installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("Tide helper functions defined.")

### Estimate the tide level at the time of every image

To use a tide model to allow you to get tides anywhere in the world anytime, first 
- download a model. 
    - For example, the EOT20 (~2GB) is one of the smaller ones, easy to use, and doesnt require an account from here: https://www.seanoe.org/data/00683/79489/ 
        - see their article here: https://essd.copernicus.org/articles/13/3869/2021
    - FES2022 is also a great and popular tide model for this work 


In [ ]:
# ---- Your turn: where do the tide heights come from? ---------------------------------------
TIDE_SOURCE  = "eo_tides"           # "noaa"     = measured water level at a NOAA tide gauge
                                # "eo_tides" = a global tide MODEL (works anywhere, needs model files, see above)
                                # "csv"      = your own file
# --- noaa gauge ---
TIDE_STATION = "1612340"        # Honolulu, Hawaii. Find other stations at https://tidesandcurrents.noaa.gov/map/
TIDE_PRODUCT = "hourly_height"  # measured, verified hourly water level (verification lags real time by months)
                                # or "predictions" = harmonic tide predictions (always available, no weather)
TIDE_DATUM   = "MSL"            # heights relative to mean sea level

# --- eo_tides model ---
TIDE_MODEL     = "EOT20"
TIDE_MODEL_DIR = "tide_models"     # the folder you made with clip_models (or the full tide_models/ folder)
TIDE_LONLAT    = None                       # (lon, lat) of the beach; None = centre of your images
# --- csv ---
TIDE_CSV = "tide_heights.csv"               # columns: time_utc, height_m  (metres above the reference level)
# --- when the image was taken, if the file name has no time of day ---
DEFAULT_UTC_TIME = "21:00"      # Hawaii: Sentinel-2 / Landsat pass at about 21:00 UTC (10-11 am local). California S2: about 18:40
# --- beach slope and reference level ---
BEACH_SLOPE = None              # tan(beta), e.g. 0.05 = 1:20.  None = estimate it from your own time series (next cells)
TIDE_REF    = 0.0               # correct all shorelines to this tide level (0 = mean sea level)
# ----------------------------------------------------------------------------------------------

default_t = pd.to_datetime(ts["date"]).dt.normalize() + pd.to_timedelta(DEFAULT_UTC_TIME + ":00")
if "time_utc" not in ts.columns:
    ts["time_utc"] = pd.NaT
ts["time_utc"] = pd.to_datetime(ts["time_utc"])
n_named = int(ts["time_utc"].notna().sum())
ts["time_utc"] = ts["time_utc"].fillna(default_t)
print(f"{n_named} of {len(ts)} images have a time of day in the file name; the others use DEFAULT_UTC_TIME = {DEFAULT_UTC_TIME} UTC")


###### Get tide values from Honolulu tide gauge (or specified tide gauge)
tide_full = None
if TIDE_SOURCE == "noaa":
    tide_full = fetch_noaa_tide(TIDE_STATION, ts["time_utc"].min().normalize() - pd.Timedelta(days=1),
                                ts["time_utc"].max().normalize() + pd.Timedelta(days=1), TIDE_PRODUCT, TIDE_DATUM)
    ts["tide_m"] = interp_tide(tide_full, ts["time_utc"])
elif TIDE_SOURCE == "csv":
    tide_full = pd.read_csv(TIDE_CSV, parse_dates=["time_utc"]).set_index("time_utc")["height_m"].sort_index()
    ts["tide_m"] = interp_tide(tide_full, ts["time_utc"])
elif TIDE_SOURCE == "eo_tides":
    lonlat = TIDE_LONLAT or image_centre_lonlat()
    print(f"Running tide model {TIDE_MODEL} at lon {lonlat[0]:.4f}, lat {lonlat[1]:.4f} ...")
    ts["tide_m"] = model_tide_series(ts["time_utc"], lonlat, TIDE_MODEL, TIDE_MODEL_DIR).values
else:
    raise ValueError("TIDE_SOURCE must be 'noaa', 'eo_tides' or 'csv'")

n_missing = int(ts["tide_m"].isna().sum())
print(f"Tide at the image times: {ts['tide_m'].min():.2f} to {ts['tide_m'].max():.2f} m (std {ts['tide_m'].std():.2f} m);  no tide value for {n_missing} of {len(ts)} images")
if n_missing > 0.05*len(ts):
    print("   Many images have no tide value. With a gauge, try TIDE_PRODUCT = 'predictions' (the measured series has gaps and lags behind real time).")

fig, axs = plt.subplots(1, 2, figsize=(12, 3.6), gridspec_kw=dict(width_ratios=[1.6, 1]))
axs[0].plot(ts["date"], ts["tide_m"], ".", ms=5)
axs[0].set_ylabel("tide at image time (m above reference)"); axs[0].set_title("The tide each image 'saw'")
if tide_full is not None:
    axs[1].hist(tide_full.dropna(), bins=40, density=True, alpha=0.5, color="gray", label="all hours")
axs[1].hist(ts["tide_m"].dropna(), bins=20, density=True, alpha=0.7, label="at image times")
axs[1].set_xlabel("tide height (m)"); axs[1].set_ylabel("density"); axs[1].legend(); axs[1].set_title("Which tides does the satellite sample?")
plt.tight_layout(); plt.show()



**Questions**: are the tides evenly sampled? do they look extremely high or low - do you think they make a difference in the shoreline position? 


- Note about tide model: (..also high probability that it wont work today...) but if you are curious check neat work of mapping tides and intertidal env with satellites and tide models: https://github.com/GeoscienceAustralia/eo-tides (with nice how-to tutorials!) - tide modeling code inspired from there..
- Side note - typically pick a location 1-2 miles offshore, the tide models may have artifacts close to shore 

#### 8a Correcting satellite shoreline position from tidal variability

We need a estimate of beach slope 
- it is rarely availble for remote sites. 
- sometimes there are old surveys or estimates you can use. 
- It also possible to estimate a beach slope from satellite-derived shorelines!!! but note, they work somewhat poorly in environments with small tidal range and steep beaches (e.g., Hawaii!)... but fairly well where there is a large tidal range. See  https://agupubs.onlinelibrary.wiley.com/doi/full/10.1029/2020gl088365


In [ ]:
#### USER DEFINED SLOPE
BEACH_SLOPE = 0.1

ts["dist_tc"] = ts["dist_m"] + (ts["tide_m"] - TIDE_REF) / BEACH_SLOPE       # move each measurement to where it would be at the reference tide
clean = ts[~ts["outlier"]].copy()
both = clean.dropna(subset=["dist_tc"])                                # compare like with like: same images in both series
print(f"{len(both)} images have both a raw and a tide-corrected position")

fig, axs = plt.subplots(2, 2, figsize=(12, 7), gridspec_kw=dict(height_ratios=[1.2, 1]))
gs = axs[0, 0].get_gridspec()
for a in axs[0, :]: a.remove()
axt = fig.add_subplot(gs[0, :])
axt.plot(both["date"], both["dist_m"], "o", color="lightgray", ms=5, label="raw")
axt.plot(both["date"], both["dist_tc"], "o", color="tab:green", ms=4, label="tide-corrected")
axt.set_ylabel("Shoreline position (m)"); axt.legend(); axt.set_title("Raw vs tide-corrected shoreline position")
for ax, col, ttl in zip(axs[1, :], ["dist_m", "dist_tc"], ["Before correction", "After correction"]):
    y = both[col] - np.polyval(np.polyfit(both["t_yr"], both[col], 1), both["t_yr"])
    ax.plot(both["tide_m"], y, ".", color="tab:blue" if col == "dist_m" else "tab:green")
    r = np.corrcoef(both["tide_m"], y)[0, 1]
    ax.set_xlabel("tide height at image time (m)"); ax.set_ylabel("position, trend removed (m)"); ax.set_title(f"{ttl}: correlation with tide = {r:.2f}")
plt.tight_layout(); plt.show()

**thoughts** could correcting for tides 'save' some of the outliers 

## Step 9: How noisy is it, and what can we actually see?
Two simple questions:
1. **One image:** how much do single measurements scatter, and how different must two images be before we believe the beach really moved?
2. **The long run:** can we still see a slow trend through that scatter?

*Noise* here means the scatter that is left **after** removing the trend and the yearly wave (the "residual scatter" from Step 7c). We measure it before and after tide correction.

In [ ]:
print(f"Total change implied by trend over the record: {beta[1]*t_c.max():.1f} m")
print(f"Noise level (robust std):                      {noise_sigma:.1f} m")
print(f"Pixel size:                                    {RES:g} m")

In [ ]:
# How noisy is one image?  Same images, before and after tide correction.
def noise_table(df, cols):
    rows = []
    L = df["t_yr"].max()                                  # record length (years)
    for label, col in cols:
        b, se, dof, fit, res = fit_harmonic(df["t_yr"].values, df["yfrac"].values, df[col].values)   # from Step 7c
        sigma = res.std()                                 # noise of ONE image (m)
        rows.append({"series": label,
                     "noise, 1 sigma (m)": sigma,
                     "one image is within +/- (m)": 1.96*sigma,          # 95% of images
                     "two images must differ by > (m)": 1.96*np.sqrt(2)*sigma,   # difference of two noisy values
                     "trend (m/yr)": b[1],
                     "trend +/- 95% (m/yr)": stats.t.ppf(0.975, dof)*se[1],
                     "total change over record (m)": b[1]*L})
    return pd.DataFrame(rows).set_index("series").round(1)

noise = noise_table(both, [("raw", "dist_m"), ("tide-corrected", "dist_tc")])
display(noise.T)
sig = noise.loc["tide-corrected", "noise, 1 sigma (m)"]
chg = abs(noise.loc["tide-corrected", "total change over record (m)"])
print(f"Pixel size: {RES:g} m.   Noise of one image: {sig:.1f} m.   Total trend change over {both['t_yr'].max():.0f} years: {chg:.1f} m.")
print(f"-> the trend over the whole record is {chg/sig:.1f} x the noise of a single image.")

In [ ]:
# What does that look like?  Left: single images.  Right: yearly averages.
fit_t = np.linspace(both["t_yr"].min(), both["t_yr"].max(), 500)
b, se, dof, fit, res = fit_harmonic(both["t_yr"].values, both["yfrac"].values, both["dist_tc"].values)
d_fit = both["date"].min() + pd.to_timedelta(fit_t*365.25, unit="D")
yf_fit = (pd.DatetimeIndex(d_fit).day_of_year.values - 1) / 365.25               # time of year
Xs = np.column_stack([np.ones_like(fit_t), fit_t, np.sin(2*np.pi*yf_fit), np.cos(2*np.pi*yf_fit)])

fig, axs = plt.subplots(1, 2, figsize=(13, 4), gridspec_kw=dict(width_ratios=[1.6, 1]))
ax = axs[0]
ax.plot(both["date"], both["dist_tc"], ".", color="tab:green", ms=6, label="single images (tide-corrected)")
ax.fill_between(d_fit, Xs @ b - 1.96*res.std(), Xs @ b + 1.96*res.std(), color="gray", alpha=0.2,
                label="where 95% of single images fall")
ax.plot(d_fit, b[0] + b[1]*fit_t, "r--", lw=2, label=f"trend: {b[1]:.2f} m/yr")
ax.set_ylabel("Shoreline position (m)"); ax.legend(fontsize=8, loc="lower right"); ax.set_title("Single images vs the trend")

# yearly mean +/- 95% interval: averaging many images beats down the noise
g = both.groupby(both["date"].dt.year)["dist_tc"]
yr = pd.DataFrame({"mean": g.mean(), "n": g.count(), "sd": g.std()})
yr = yr[yr["n"] >= 5]                                   # skip years with very few images
yr["ci"] = stats.t.ppf(0.975, yr["n"] - 1)*yr["sd"]/np.sqrt(yr["n"])
axs[1].errorbar(yr.index, yr["mean"], yerr=yr["ci"], fmt="o-", capsize=4, color="tab:green")
axs[1].set_xlabel("year"); axs[1].set_ylabel("mean position (m)")
axs[1].set_title("Yearly averages (+/- 95%)")
plt.tight_layout(); plt.show()
print(f"Typical uncertainty of a yearly average: +/- {yr['ci'].median():.1f} m  (from about {int(yr['n'].median())} images per year)")

### bonus: How much does the threshold matter? what shoreline are we defining
Rerun the whole extraction three ways: Otsu (automatic), a fixed NDWI = 0.0 (the textbook value) and a fixed NDWI = 0.3.

In [ ]:
variants = {"Otsu (automatic)": None, "fixed 0.0": 0.0, "fixed 0.3": 0.3}
fig, ax = plt.subplots(figsize=(11, 4))
cmp = {}
for (label, thr), colr in zip(variants.items(), ["tab:blue", "tab:orange", "tab:green"]):
    tv = build_timeseries(scenes, TRANSECT, threshold=thr).set_index("date")["dist_m"]
    cmp[label] = tv
    ax.plot(tv.index, tv.values, "o-", ms=3, lw=0.8, color=colr, label=label)
ax.set_ylabel("Shoreline position (m)"); ax.legend(); ax.set_title("Same images, different threshold")
plt.show()

cmp = pd.DataFrame(cmp)
print("Difference from Otsu (m):")
print((cmp.sub(cmp["Otsu (automatic)"], axis=0)).drop(columns="Otsu (automatic)").describe().loc[["mean", "std"]].round(1))

**Discuss:** the threshold shifts the *level* of the line (a constant offset) much more than it changes the *shape* of the time series. Why does that make satellite shorelines good at measuring **change** even when absolute position is uncertain?

## Discussion: what is real and what is artefact?
Look back at your time series and decide, for each feature, whether it is a **real change** or an **artefact** of the method:

- A single point far from its neighbours.
- A constant offset between the Otsu and fixed-threshold results.
- The rise and fall each year.
- A sudden retreat followed by slow recovery.
- The overall slope.

**Things the satellite can't see:** tide level at the moment of the image, wave runup, wet vs dry sand, sub-pixel detail. Sentinel-2 pixels are 10 m, so changes of a few metres per year only become visible once you have several years of data.

**Where to go next**
- Multiple transects along the beach (alongshore variability).
- Correct for tide (the main source of noise on gently sloping beaches).
- Sub-pixel shoreline extraction and cloud masking with Sentinel-2's QA bands.
- [CoastSat](https://github.com/kvos/CoastSat) is the established toolkit that does all this at scale.
- See README in github for more info and ressources: https://github.com/abmikkelsen/coastal_mapping_workshop/